# RAG Corpus Sampling

Produces `data/rag_corpus.jsonl` -- the 12,000-record retrieval corpus used for ChromaDB ingestion.

**Method:** Proportional stratified random sample across all four severity levels (CRITICAL, HIGH, MEDIUM, LOW). Each stratum is sampled in proportion to its share of the 156k quality-filtered pool, so the corpus reflects the real-world severity distribution. Fixed seed (`SEED = 42`) ensures reproducibility.

Enrichment (EPSS, KEV) is handled in a separate notebook.

In [1]:
import json
import random
import pandas as pd
from pathlib import Path
from collections import Counter

SEED = 42
TARGET_TOTAL = 12_000
DATA_DIR = Path("../data")

SEVERITY_FILES = {
    "CRITICAL": DATA_DIR / "cves_all_critical.jsonl",
    "HIGH":     DATA_DIR / "cves_all_high.jsonl",
    "MEDIUM":   DATA_DIR / "cves_all_medium.jsonl",
    "LOW":      DATA_DIR / "cves_all_low.jsonl",
}

## 1. Load quality-filtered pool

In [2]:
def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

by_severity = {sev: load_jsonl(path) for sev, path in SEVERITY_FILES.items()}

pool_counts = {sev: len(records) for sev, records in by_severity.items()}
total_pool = sum(pool_counts.values())

print(f"Quality-filtered pool ({total_pool:,} total):")
for sev, count in pool_counts.items():
    print(f"  {sev:<10} {count:>7,}  ({count / total_pool * 100:.1f}%)")

Quality-filtered pool (156,084 total):
  CRITICAL    17,629  (11.3%)
  HIGH        57,397  (36.8%)
  MEDIUM      74,404  (47.7%)
  LOW          6,654  (4.3%)


## 2. Compute proportional sample targets

In [3]:
def proportional_targets(pool_counts, target_total):
    """Floor each stratum, then give the remainder to strata with the largest fractional parts."""
    total = sum(pool_counts.values())
    raw = {sev: target_total * count / total for sev, count in pool_counts.items()}
    floored = {sev: int(v) for sev, v in raw.items()}
    remainder = target_total - sum(floored.values())
    by_fraction = sorted(raw, key=lambda s: -(raw[s] % 1))
    for sev in by_fraction[:remainder]:
        floored[sev] += 1
    return floored

targets = proportional_targets(pool_counts, TARGET_TOTAL)

print(f"Sample targets (total = {sum(targets.values()):,}):")
for sev, n in targets.items():
    print(f"  {sev:<10} {n:>6,}  ({n / TARGET_TOTAL * 100:.1f}%)")

Sample targets (total = 12,000):
  CRITICAL    1,355  (11.3%)
  HIGH        4,413  (36.8%)
  MEDIUM      5,720  (47.7%)
  LOW           512  (4.3%)


## 3. Sample within each stratum

In [4]:
rng = random.Random(SEED)

sampled = []
for sev, records in by_severity.items():
    n = targets[sev]
    stratum_sample = rng.sample(records, n)
    sampled.extend(stratum_sample)

rng.shuffle(sampled)

print(f"Corpus size: {len(sampled):,} records")

Corpus size: 12,000 records


## 4. Distribution checks

In [5]:
# Severity breakdown
sev_counts = Counter(r["cvss_severity"] for r in sampled)
print("Severity breakdown:")
for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    n = sev_counts[sev]
    print(f"  {sev:<10} {n:>6,}  ({n / len(sampled) * 100:.1f}%)")

Severity breakdown:
  CRITICAL    1,355  (11.3%)
  HIGH        4,413  (36.8%)
  MEDIUM      5,720  (47.7%)
  LOW           512  (4.3%)


In [6]:
# Year distribution
years = [r["published"][:4] for r in sampled]
year_counts = Counter(years)

df_years = pd.DataFrame(
    sorted(year_counts.items()),
    columns=["year", "count"]
)
df_years["pct"] = (df_years["count"] / len(sampled) * 100).round(1)
print("Year distribution:")
print(df_years.to_string(index=False))

Year distribution:
year  count  pct
2020    999  8.3
2021   1410 11.8
2022   1707 14.2
2023   1954 16.3
2024   2288 19.1
2025   2284 19.0
2026   1358 11.3


In [7]:
# CPE vendor distribution
def extract_vendor(record):
    try:
        cpe = record["configurations"][0]["nodes"][0]["cpeMatch"][0]["criteria"]
        parts = cpe.split(":")
        return parts[3] if len(parts) > 3 else "unknown"
    except (KeyError, IndexError):
        return "unknown"

vendors = [extract_vendor(r) for r in sampled]
vendor_counts = Counter(vendors)

top20 = pd.DataFrame(vendor_counts.most_common(20), columns=["vendor", "count"])
top20["pct"] = (top20["count"] / len(sampled) * 100).round(1)

print(f"Unique vendors in corpus: {len(vendor_counts):,}")
print(f"\nTop 20 vendors (of {len(vendor_counts):,} total):")
print(top20.to_string(index=False))

Unique vendors in corpus: 3,574

Top 20 vendors (of 3,574 total):
    vendor  count  pct
     linux    885  7.4
    google    661  5.5
     adobe    274  2.3
     apple    273  2.3
       ibm    241  2.0
    oracle    236  2.0
 microsoft    209  1.7
     cisco    167  1.4
    apache    138  1.2
     tenda    137  1.1
    huawei    119  1.0
   siemens    100  0.8
     dlink     96  0.8
   jenkins     96  0.8
     intel     94  0.8
phpgurukul     93  0.8
      dell     90  0.8
   samsung     90  0.8
   mozilla     90  0.8
    gitlab     85  0.7


## 5. Additional corpus analysis

In [ ]:
# Attack vector distribution
av_counts = Counter(r["attack_vector"] for r in sampled)

print("Attack vector distribution (overall):")
for av, count in sorted(av_counts.items(), key=lambda x: -x[1]):
    print(f"  {av:<20} {count:>6,}  ({count / len(sampled) * 100:.1f}%)")

print("\nAttack vector by severity (%):")
avs = ["NETWORK", "LOCAL", "ADJACENT_NETWORK", "PHYSICAL"]
print(f"  {'':>10}  " + "  ".join(f"{av:<18}" for av in avs))
for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    sev_records = [r for r in sampled if r["cvss_severity"] == sev]
    sev_total = len(sev_records)
    av_sev = Counter(r["attack_vector"] for r in sev_records)
    row = f"  {sev:<10}  "
    row += "  ".join(f"{av_sev.get(av, 0) / sev_total * 100:>5.1f}% ({av_sev.get(av, 0):>5,})" for av in avs)
    print(row)

In [ ]:
# CVSS score distribution within severity bands
print("CVSS score distribution within severity bands:")
for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    scores = pd.Series([r["cvss_score"] for r in sampled if r["cvss_severity"] == sev])
    print(f"\n  {sev} (n={len(scores):,}):")
    print(f"    min={scores.min():.1f}  median={scores.median():.1f}  mean={scores.mean():.2f}  max={scores.max():.1f}")
    bins = scores.value_counts(bins=5, sort=False).sort_index()
    for interval, cnt in bins.items():
        print(f"    {str(interval):<25} {cnt:>5,}  ({cnt / len(scores) * 100:.1f}%)")

In [ ]:
# Description length distribution
desc_lengths = pd.Series([len(r["description"]) for r in sampled])

print("Description length (characters) -- full corpus:")
print(f"  min={desc_lengths.min()}  median={desc_lengths.median():.0f}  mean={desc_lengths.mean():.0f}  max={desc_lengths.max()}")
print(f"  25th pct={desc_lengths.quantile(0.25):.0f}  75th pct={desc_lengths.quantile(0.75):.0f}  99th pct={desc_lengths.quantile(0.99):.0f}")

print("\nDescription length by severity:")
for sev in ["CRITICAL", "HIGH", "MEDIUM", "LOW"]:
    s = pd.Series([len(r["description"]) for r in sampled if r["cvss_severity"] == sev])
    print(f"  {sev:<10}  median={s.median():.0f}  mean={s.mean():.0f}  min={s.min()}  max={s.max()}")

print("\nLength buckets (whole corpus):")
buckets = [(100, 200), (200, 400), (400, 700), (700, 1000), (1000, 9999)]
for lo, hi in buckets:
    count = ((desc_lengths >= lo) & (desc_lengths < hi)).sum()
    label = f"{lo}-{hi-1}" if hi != 9999 else f"{lo}+"
    print(f"  {label:<12} {count:>6,}  ({count / len(desc_lengths) * 100:.1f}%)")

## 6. Save corpus

In [8]:
output_path = DATA_DIR / "rag_corpus.jsonl"

with open(output_path, "w") as f:
    for record in sampled:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(sampled):,} records to {output_path}")

Saved 12,000 records to ../data/rag_corpus.jsonl
